In [ ]:
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()
await page.goto("https://www.craigslist.org/area/sandiego")
from pathlib import Path

scan_js_path = "../navgent/js/scan-page.js"
js_script = Path(scan_js_path).read_text()
await page.main_frame.evaluate(js_script)


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict


class Navigate(BaseModel):
    model_config = ConfigDict(extra="forbid")

    action: Literal["navigate"]
    url: str


class AgentStep(BaseModel):
    model_config = ConfigDict(extra="forbid")

    tool: Navigate


In [ ]:
async def click():
    pass


async def navigate(url: str):
    await page.goto(url)


In [15]:
import json

SYSTEM_PROMPT = """
        You are a browser agent.
        Choose the next action.

        navigate: provide a URL.
        """


def determine_step(messages: list) -> AgentStep:
    response = client.responses.parse(
        model="gpt-4.1-mini",
        instructions=SYSTEM_PROMPT,
        input=messages,
        text_format=AgentStep,
    )
    if response.output_parsed is None:
        raise RuntimeError("No structured action returned.")
    return response.output_parsed


async def execute_step(step: AgentStep):
    if step.tool.action == "navigate":
        await navigate(step.tool.url)
        return {"status": "success"}
    return None


messages = [
    {"role": "user", "content": "Navigate to https://example.com"}
]

MAX_ITERATIONS = 1
current_iteration = 0

while current_iteration <= MAX_ITERATIONS:
    step = determine_step(messages)
    result = await execute_step(step)
    current_iteration += 1


In [ ]:
await browser.close()
await playwright.stop()
